#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date

#Get Data From Bronze

In [0]:
# Load ERP customer data from Azure Data Lake Storage
df = spark.read.format("parquet")\
        .option("inferSchema",True)\
        .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12")
df.limit(10).display()

CID,BDATE,GEN
NASAW00011000,1971-10-06,Male
NASAW00011001,1976-05-10,Male
NASAW00011002,1971-02-09,Male
NASAW00011003,1973-08-14,Female
NASAW00011004,1979-08-05,Female
NASAW00011005,1976-08-01,Male
NASAW00011006,1976-12-02,Female
NASAW00011007,1969-11-06,Male
NASAW00011008,1975-07-04,Female
NASAW00011009,1969-09-29,Male


#Data Transformation

##Trimming

In [0]:
# Remove leading and trailing spaces from string columns
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

##Customer ID Cleanup

In [0]:
# Remove the NAS prefix from customer IDs when present
df = df.withColumn(
    "CID",when(
        col("CID").like("NAS%"),
        substring(col("CID"),4,length(col("CID")))
    ).otherwise(col("CID"))
)

In [0]:
df.limit(10).display()

CID,BDATE,GEN
AW00011000,1971-10-06,Male
AW00011001,1976-05-10,Male
AW00011002,1971-02-09,Male
AW00011003,1973-08-14,Female
AW00011004,1979-08-05,Female
AW00011005,1976-08-01,Male
AW00011006,1976-12-02,Female
AW00011007,1969-11-06,Male
AW00011008,1975-07-04,Female
AW00011009,1969-09-29,Male


##Birthdate Validation

In [0]:
# Replace future birthdates with null
df = df.withColumn(
    "BDATE",when(
        col("BDATE") > current_date(),
        None
    ).otherwise(col("BDATE"))
)

In [0]:
# Check for any remaining future birthdates
df.filter(col("BDATE") > current_date()).limit(10).display()

CID,BDATE,GEN


##Gender Normalization

In [0]:
# Standardize gender values and handle unknown entries
df = df.withColumn(
        "GEN", when(
            (upper(col("GEN")) == "M") | (upper(col("GEN")) == "MALE"),
            "Male"
        ).when(
            (upper(col("GEN")) == "F") | (upper(col("GEN")) == "FEMALE"),
            "Female"
        ).otherwise("n/a")
    )

In [0]:
# Verify standardized gender categories
df.select("GEN").distinct().display()

GEN
Female
n/a
Male


##Renaming Columns

In [0]:
# Rename source columns to business-friendly names
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

##Create silver record_priority to filter out duplicate id

In [0]:
# Rank customer records by latest birthdate
w = Window.partitionBy(df["customer_number"]).orderBy(df["birth_date"].desc_nulls_last())
df = df.withColumn("record_priority",row_number().over(w))

In [0]:
# Keep the highest-priority record for each customer
df = df.filter(col("record_priority") == 1)

#Data Writing

In [0]:
# Save cleaned customer data in Parquet format
df.write.format("parquet")\
    .mode("overwrite")\
    .option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12` LIMIT 10

customer_number,birth_date,gender,record_priority
AW00011000,1971-10-06,Male,1
AW00011001,1976-05-10,Male,1
AW00011002,1971-02-09,Male,1
AW00011003,1973-08-14,Female,1
AW00011004,1979-08-05,Female,1
AW00011005,1976-08-01,Male,1
AW00011006,1976-12-02,Female,1
AW00011007,1969-11-06,Male,1
AW00011008,1975-07-04,Female,1
AW00011009,1969-09-29,Male,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_CUST_AZ12` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

customer_number,birth_date,gender,record_priority
